In [1]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter

video_id = "jzD_yyEcp0M"  # Lex Fridman interview with Demis Hassabis

try:
    print("Connecting to YouTube to retrieve captions...")
    yt_api = YouTubeTranscriptApi()
    
    # Modern object-oriented call strategy to retrieve captions array
    raw_transcript = yt_api.fetch(video_id, languages=["en"])
    
    # Extract structural text strings via object properties (.text)
    transcript = " ".join(segment.text for segment in raw_transcript)
    print(f"Success! Transcript length: {len(transcript)} characters.")

except TranscriptsDisabled:
    print("Error: Transcripts are disabled for this specific YouTube video.")
    
# Structural Text Splitting configuration
print("Splitting video text into processing chunks...")
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])
print(f"Total Text Document Chunks created: {len(chunks)}")

# --- Added: Printing the full transcript at the end ---
print("\n" + "="*50)
print("FULL VIDEO TRANSCRIPT:")
print("="*50)
print(transcript)
print("="*50)

Connecting to YouTube to retrieve captions...
Success! Transcript length: 559 characters.
Splitting video text into processing chunks...
Total Text Document Chunks created: 1

FULL VIDEO TRANSCRIPT:
[Music] [Applause] you say you love me i say you crazy with nothing more than friends you're not my lover more like a brother i know you since we were like 10. don't mess it up talking that only gonna push me away that's it when you say you love me that make me crazy [Music] [Music] yes [Music] [Applause] [Music] have you got no shame you looking insane turning up at my door in the morning [Music] [Music] yes [Applause] [Music] get that inside your head we're just friends [Music] [Music] [Music] times [Music] is oh what are you doing come on [Music] you


In [6]:
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama

print("Connecting to local Ollama instance...")

# Initialize local embedding vectors using your Qwen local runner setup
embeddings = OllamaEmbeddings(model="qwen2.5-coder:3b")

# Initialize local Chat generation LLM instance using your Qwen local runner setup
llm = ChatOllama(model="qwen2.5-coder:3b", temperature=0.2)

# Compile your pieces directly into a local FAISS DB instance
print("Generating context vectors and building local FAISS storage matrix...")
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
print("Local database indexing sequence finished successfully!")

Connecting to local Ollama instance...
Generating context vectors and building local FAISS storage matrix...
Local database indexing sequence finished successfully!


In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# Transformation helper function to strip and join document fragments cleanly
def format_docs(retrieved_docs):
    return "\n\n".join(doc.page_content for doc in retrieved_docs)

# Rigorous context constraint prompt template structure
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables=['context', 'question']
)

# A context mapping blueprint used to preview underlying metadata values
parallel_context_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

# End-To-End operational RAG processing chain architecture
rag_pipeline = (
    RunnableParallel({
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    })
    | prompt 
    | llm 
    | StrOutputParser()
)
print("Pipeline blueprints constructed. Systems online and ready for testing.")

Pipeline blueprints constructed. Systems online and ready for testing.


In [5]:
# Execution Query 1: Verify document contextual structural parsing
print("--- Execution Step: Diagnostic Context Extraction ---")
# Changed the diagnostic string to match the music topic
test_payload = parallel_context_chain.invoke('what is the relationship between them?')
print("Successfully verified retrieval payload mapping format!")

print("\n" + "="*50 + "\n")

# Execution Query 2: Local AI Generation Processing using offline Qwen
print("--- Execution Step: Local Querying over RAG Pipeline ---")

# UPDATED: Replaced the nuclear fusion question with relevant lyric questions
target_question = "What does the text say about the two people being lovers? How long have they known each other?"
system_response = rag_pipeline.invoke(target_question)

print("Local Qwen Assistant Response:")
print("-" * 50)
print(system_response)
print("-" * 50)

--- Execution Step: Diagnostic Context Extraction ---
Successfully verified retrieval payload mapping format!


--- Execution Step: Local Querying over RAG Pipeline ---
Local Qwen Assistant Response:
--------------------------------------------------
The text says that the two people are not lovers, but rather friends. They have been friends since they were 10 years old.
--------------------------------------------------
